# 01. Hello World with Google AI Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/01_ai_studio_quickstart.ipynb)

This cookbook uses `[jupyter].setup_packages` in `drawlib.toml` to automatically prepend `!pip install -q drawlib google-genai pydantic` at the top of every generated notebook, so zero manual package installation is required in Google Colab. To run your first Gemini call safely without hardcoding credentials in Git, generate an API key (`GEMINI_API_KEY`) in **Google AI Studio**—backed by a **Google Cloud Project**—and store it in Google Colab's **Secrets** sidebar (`🔑` icon on the left pane with **Notebook access** toggled ON).

![Figure 1-1: End-to-End Google AI Studio & Colab Secrets Setup Workflow](https://examples.drawlib.com/google-genai/images/01_ai_studio_quickstart/ch01_aistudio_workflow.png)

*Figure 1-1: End-to-End Google AI Studio & Colab Secrets Setup Workflow*

## 1. Minimal Hello World (`userdata` + `genai.Client`)

Before running the code cell below in Google Colab, complete this one-time secret setup:

1. Open **Google AI Studio** (`aistudio.google.com/apikey`), click **Create API key**, and select an existing or new **Google Cloud Project**.
2. In Google Colab, click the **Secrets (`🔑`)** icon in the left sidebar, add a secret named `GEMINI_API_KEY`, paste your key value, and toggle **Notebook access** ON.
3. Read the key at runtime via `userdata.get("GEMINI_API_KEY")` and pass it directly to `genai.Client`:

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain in two sentences why unified SDKs improve developer productivity.",
)
print(response.text)

`genai.Client` is the single entry point across the entire `google-genai` SDK. Every `client.models.*` and `client.chats.*` call you learn in this chapter works identically whether you authenticate with an AI Studio API key or Google Cloud Vertex AI IAM credentials.

## 2. Google Cloud Project Linkage: Free Tier vs. Paid Tier

Every API key issued in Google AI Studio belongs to an underlying **Google Cloud Project**. If you already work with Google Cloud, you can select any existing project directly inside AI Studio's key creation dialog—allowing IAM owners and cloud administrators to govern API enablement (`generativelanguage.googleapis.com`), quota monitoring, and billing in one place:

- **Free Tier (Unbilled Project)**: When the selected Google Cloud Project has **no Cloud Billing account attached**, your API key runs on the Free Tier at zero cost. This is ideal for hackathons, classroom labs, and quick notebook experiments, with lower request/token rate quotas (RPM/TPM/RPD) and standard Free Tier terms where prompts and responses may be used to improve Google products.
- **Paid Tier (Cloud Billing Enabled)**: Enabling **Cloud Billing** on that exact same Google Cloud Project upgrades your `GEMINI_API_KEY` to Pay-as-You-Go token pricing. Paid Tier keys unlock higher production RPM/TPM quotas and **enterprise-grade data privacy**—your prompts and model outputs are **not** used to train Google models—with zero changes to your Python code.

![Figure 1-2: Google AI Studio Free Tier vs. Cloud Billing Paid Tier](https://examples.drawlib.com/google-genai/images/01_ai_studio_quickstart/ch01_billing_tiers.png)

*Figure 1-2: Google AI Studio Free Tier vs. Cloud Billing Paid Tier*

## 3. `GenerateContentConfig` & Inline Token Visualization with Drawlib

To control model persona, determinism, and output formatting, pass a `types.GenerateContentConfig` object to `generate_content`. Every response also attaches `response.usage_metadata`, exposing exact token counts across your input prompt (`prompt_token_count`), internal reasoning (`thoughts_token_count`), and final answer (`candidates_token_count`):

In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    system_instruction="You are a concise cloud architect. Answer in three short bullet points.",
    temperature=0.2,
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="List three security best practices for managing API keys in notebooks.",
    config=config,
)
print(response.text)

usage = response.usage_metadata
prompt_tokens = usage.prompt_token_count or 0
thoughts_tokens = usage.thoughts_token_count or 0
candidates_tokens = usage.candidates_token_count or 0

print(f"Prompt tokens:     {prompt_tokens}")
print(f"Thoughts tokens:   {thoughts_tokens}")
print(f"Candidates tokens: {candidates_tokens}")

Because `drawlib` is pre-installed in the notebook alongside `google-genai`, you can render those live `usage_metadata` metrics directly inside the next Colab output cell using `setup(..., cwd=True)`, `BarChart`, and `show()`:

In [ ]:
from drawlib.canvas import setup, show
from drawlib.charts.bar import BarChart
from drawlib.styles import Styles

setup(width=110, height=52, cwd=True)

chart = BarChart(
    axis_line_style=Styles.MutedDashed,
    categories=["Prompt", "Thoughts", "Candidates"],
    width=96.0,
    height=40.0,
    title="Gemini 2.5 Flash — Live Token Usage Breakdown",
    title_style=Styles.DarkBold.patch(text_size=12.0),
    background_style=Styles.Neutral.patch(shape_r=2.0),
    bar_width_ratio=0.48,
    bar_r=0.8,
    axis_text_style=Styles.Dark.patch(text_size=10.5),
    grid_style=Styles.MutedThin,
    value_text_style=Styles.DarkBold.patch(text_size=10.5),
    value_format="{:.0f}",
)
chart.add_series(
    "Tokens",
    [float(prompt_tokens), float(thoughts_tokens), float(candidates_tokens)],
    style=Styles.PrimaryFlat,
)
chart.configure_y_axis(min_value=0.0)
chart.draw(xy=(7.0, 6.0))

show()

### Key Takeaways & Next Step

- **Zero-Friction Setup**: Storing `GEMINI_API_KEY` in Colab Secrets keeps notebooks clean, shareable, and safe to commit to Git while linking directly to your Google Cloud Project's Free or Paid Tier.
- **Up Next in Chapter 02**: Learn how to switch `genai.Client` to **Google Cloud Vertex AI** (`vertexai=True`) using keyless IAM authentication while keeping 100% of your downstream Gemini code unchanged.